<a href="https://colab.research.google.com/github/leeroouo/earth-Geophysics/blob/main/hw2_geophysics_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import importlib.util
import subprocess
import sys

# 在子程序檢查，避免安裝前先載入目前 kernel 的動態函式庫
check_code = """
import pygmt, pandas
import shutil
from pygmt.clib import Session
assert pygmt.__version__.lstrip('v').startswith('0.17.')
assert shutil.which('gs')
with Session() as session:
    assert session.info['version'].startswith('6.5.')
"""
try:
    ENV_READY = subprocess.run(
        [sys.executable, "-c", check_code], capture_output=True, timeout=30
    ).returncode == 0
except subprocess.TimeoutExpired:
    ENV_READY = False

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ENV_READY:
    print("環境已可用，跳過安裝。")
elif IN_COLAB:
    print("步驟 1/2：安裝 Conda（約 1 分鐘）。完成後 Colab 會自動重啟執行環境，等重新連線再執行下一格。", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")


步驟 1/2：安裝 Conda（約 1 分鐘）。完成後 Colab 會自動重啟執行環境，等重新連線再執行下一格。
✨🍰✨ Everything looks OK!


In [14]:
import sys
!mamba install pygmt==0.17.0 gmt=6.5.0 -c conda-forge -c defaults --yes

warning  libmamba 'repo.anaconda.com', a commercial channel hosted by Anaconda.com, is used.
    
warning  libmamba Please make sure you understand Anaconda Terms of Services.
    
warning  libmamba See: https://legal.anaconda.com/policies/en/
[+] 0.0s
[+] 0.1s
conda-forge/noarch  ⣾  
pkgs/r/noarch       ⣾  
Pinned packages:

  - python=3.13

Pinned packages:

  - python_abi[version="=3.13",build="*cp313*"]

Pinned packages:

  - cuda-version*.*

Pinned packages:

  - python=3.13


Transaction

  Prefix: /usr/local

  All requested packages already installed


Transaction starting
[+] 0.0s

Transaction finished



In [22]:
import urllib.request

import numpy as np
import pandas as pd
import pygmt

# ==== 改這裡：從 plate-boundaries.md 挑一段，或自己框 ====
REGION = [-28, -10, 62, 68]        # 西、東、南、北（度），調整為冰島範圍
START, MINMAG = "2007-08-08", 2.0   # 地震起始日與最低規模
A, B = (REGION[0], 64.5), (REGION[1], 64.5)     # 剖面兩端（經度、緯度） - 調整為合適的冰島內緯度
HALF_WIDTH_KM = 100                 # 走廊半寬（km）
DEPTH_MAX = 20                    # 剖面深度軸下限
OUT_MAP, OUT_SECTION = "region_map.png", "region_section.png"
# ======================================================

# 1. 地震：先查筆數，超過 20,000 自動提高規模門檻
base = "https://earthquake.usgs.gov/fdsnws/event/1/"
minmag = MINMAG
while True:
    query = (f"starttime={START}&minmagnitude={minmag}"
             f"&minlongitude={REGION[0]}&maxlongitude={REGION[1]}&minlatitude={REGION[2]}&maxlatitude={REGION[3]}")
    count = int(urllib.request.urlopen(base + "count?" + query).read())
    if count <= 20000:
        break
    minmag += 0.5
quakes = pd.read_csv(base + "query?format=csv&" + query).dropna(subset=["longitude", "latitude", "depth", "mag"])
print(f"USGS {START} 起 M >= {minmag}：{len(quakes)} 筆")

# 2. 走廊內地震投影到 A–B 線
if quakes.empty:
    # 防錯機制 1：若根本沒有地震，直接建立空的投影 DataFrame
    selected = pd.DataFrame(columns=["longitude", "latitude", "distance", "offset", "depth", "mag"])
else:
    selected = pygmt.project(data=quakes[["longitude", "latitude", "depth", "mag"]],
                             center=list(A), endpoint=list(B), unit=True, length="w",
                             width=[-HALF_WIDTH_KM, HALF_WIDTH_KM], convention="xypqz")
    # 防錯機制 2：若有地震但投影後是空的
    if selected.empty:
        selected = pd.DataFrame(columns=["longitude", "latitude", "distance", "offset", "depth", "mag"])
    else:
        selected.columns = ["longitude", "latitude", "distance", "offset", "depth", "mag"]

selected = selected.sort_values("depth", ascending=False)
track = pygmt.project(center=list(A), endpoint=list(B), generate=5, unit=True)
track.columns = ["lon", "lat", "distance"]
length_km = float(track.distance.iloc[-1])
print(f"走廊內 {len(selected)} 筆；A–B 長 {length_km:.0f} km")

# 3. 地形
BUFFER = 0.5
GRID_REGION = [REGION[0] - BUFFER, REGION[1] + BUFFER, REGION[2], REGION[3] + BUFFER]
grid = pygmt.datasets.load_earth_relief(resolution="02m", region=GRID_REGION)
topo = pygmt.grdtrack(points=track[["lon", "lat"]], grid=grid, newcolname="z")
topo["distance"] = track["distance"]

# 4. 走廊外框
def bearing(lon1, lat1, lon2, lat2):
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(np.subtract(lon2, lon1))
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return np.degrees(np.arctan2(x, y))


def move(lon, lat, azimuth_deg, distance_km):
    p1, l1, az, d = np.radians(lat), np.radians(lon), np.radians(azimuth_deg), distance_km / 6371.0
    p2 = np.arcsin(np.sin(p1) * np.cos(d) + np.cos(p1) * np.sin(d) * np.cos(az))
    l2 = l1 + np.arctan2(np.sin(az) * np.sin(d) * np.cos(p1), np.cos(d) - np.sin(p1) * np.sin(p2))
    return np.degrees(l2), np.degrees(p2)


lon, lat = track.lon.to_numpy(), track.lat.to_numpy()
local_azimuth = bearing(lon[:-1], lat[:-1], lon[1:], lat[1:])
local_azimuth = np.append(local_azimuth, local_azimuth[-1])
left_lon, left_lat = move(lon, lat, local_azimuth - 90, HALF_WIDTH_KM)
right_lon, right_lat = move(lon, lat, local_azimuth + 90, HALF_WIDTH_KM)
corridor_lon = np.concatenate([left_lon, right_lon[::-1]])
corridor_lat = np.concatenate([left_lat, right_lat[::-1]])
# Removed: corridor_lat = np.clip(corridor_lat, -90, -60) # 限制在極區合理範圍內, no longer needed for Iceland


def mag_size(m):
    return min(0.05 * 2 ** (m - 5), 0.4)


# 5. 地圖 (墨卡托投影)
fig = pygmt.Figure()
fig.grdimage(grid=grid, region=REGION, projection="M12c", cmap="geo", shading="+a-45+nt0.5", # Changed projection to Mercator
             frame=["af", f"+tUSGS M>={minmag} since {START[:4]}"])
fig.coast(shorelines="0.3p,gray20", resolution="i")
pygmt.makecpt(cmap="#d7191c,#fdae61,#2c7bb6", series="0,70,300,700")
if not quakes.empty:
    fig.plot(x=quakes.longitude, y=quakes.latitude, size=quakes.mag.apply(mag_size) * 0.7,
             fill=quakes.depth, cmap=True, style="c", pen="0.1p,black", transparency=40)
fig.plot(x=corridor_lon, y=corridor_lat, close=True, pen="0.8p,black,--")
fig.plot(x=[A[0], B[0]], y=[A[1], B[1]], pen="1.5p,black")
fig.text(x=[A[0], B[0]], y=[A[1], B[1]], text=["A", "B"], font="10p,Helvetica-Bold",
         fill="white", pen="0.5p,black", offset="0c/0.3c")
fig.colorbar(position="JBC+w8c/0.3c+h+o0c/0.8c", frame=["a0", "+lDepth (km): 0-70 / 70-300 / 300-700"])
with fig.inset(position="jTR+w2.5c+o0.1c"):
    fig.coast(region="g", projection="G0/0/2.5c", land="gray70", water="white", frame="g") # Changed inset projection
    fig.plot(x=[REGION[0], REGION[1], REGION[1], REGION[0]], y=[REGION[2], REGION[2], REGION[3], REGION[3]],
             close=True, pen="1p,red")
fig.savefig(OUT_MAP, dpi=150)

# 6. 剖面
fig = pygmt.Figure()
fig.basemap(region=[0, length_km, -11, 9], projection="X14c/2c", frame=["Wsne", "ya5f1+lTopo (km)"])
fig.plot(x=topo.distance, y=topo.z / 1000, pen="0.8p,black")
fig.plot(x=[0, length_km], y=[0, 0], pen="0.3p,gray50,--")
fig.text(text="A", position="TL", offset="0.15c/-0.1c", font="10p,Helvetica-Bold")
fig.text(text="B", position="TR", offset="-0.15c/-0.1c", font="10p,Helvetica-Bold")
fig.shift_origin(yshift="-8.3c")
fig.basemap(region=[0, length_km, 0, DEPTH_MAX], projection="X14c/-8c",
            frame=["WSne", "xa200f100+lDistance from A (km)", "ya100f50+lDepth (km)"])
pygmt.makecpt(cmap="#d7191c,#fdae61,#2c7bb6", series="0,70,300,700")
if not selected.empty:
    fig.plot(x=selected.distance, y=selected.depth, size=selected.mag.apply(mag_size),
             fill=selected.depth, cmap=True, style="c", pen="0.3p,black", transparency=20)
ve = (8 / DEPTH_MAX) / (14 / length_km)
fig.text(text=f"corridor +-{HALF_WIDTH_KM} km, {len(selected)} events, VE = {ve:.1f}x (topo panel separate)",
         position="BL", offset="0.15c/0.15c", font="8p", fill="white@30")
fig.savefig(OUT_SECTION, dpi=150)

fixed = selected.depth.round(1).isin([10.0, 33.0, 35.0]).mean() if not selected.empty else 0
print(f"圖說骨架：USGS {START} 起 M >= {minmag}，範圍 {REGION}，A={A}、B={B}，走廊全寬 {2 * HALF_WIDTH_KM} km；"
      f"剖面 VE = {ve:.1f}x；走廊內 {fixed:.0%} 的深度是 USGS 預設值（10／33 km）。")
print("saved", OUT_MAP, OUT_SECTION)


USGS 2007-08-08 起 M >= 2.0：845 筆
走廊內 699 筆；A–B 長 859 km
圖說骨架：USGS 2007-08-08 起 M >= 2.0，範圍 [-28, -10, 62, 68]，A=(-28, 64.5)、B=(-10, 64.5)，走廊全寬 200 km；剖面 VE = 24.5x；走廊內 42% 的深度是 USGS 預設值（10／33 km）。
saved region_map.png region_section.png
